# Titeseq barcode aggregation

This notebook reads in the barcode-level counts, barcode-variant mapping, and sequencing runs total cells, so that we can prep Titeseq data for fitting by:

1. Merging in the variants substitution annotations to barcodes.
2. Normalize the counts by the total cells in each sequencing run.
3. Aggregate the counts by the variant substitution annotations.
4. Filter out low count barcodes and variants.  

## Set up analysis

In [1]:
import warnings
import yaml
import os

import numpy as np
import pandas as pd
from IPython.display import display, HTML

Ignore warnings that clutter output:

In [2]:
warnings.simplefilter('ignore')

### Parameters for notebook
Read the configuration file:

In [3]:
with open('config.yaml') as f:
    config = yaml.safe_load(f)

Make output directory if needed:

In [4]:
os.makedirs(config['counts_dir'], exist_ok=True)

## Data

Load tite-seq data

In [5]:
barcode_runs = pd.read_csv(config['barcode_runs']).drop(columns=["R1"])
barcode_runs.query("sample.str.startswith('TiteSeq')", inplace=True)
barcode_runs.set_index(["library", "sample"], inplace=True)
display(HTML(barcode_runs.head().to_html(index=True)))
print(barcode_runs.shape)

(72, 5)


In [6]:
variant_counts = pd.read_csv(config['variant_counts_file'])
variant_counts.query("sample.str.startswith('TiteSeq')", inplace=True)
display(HTML(variant_counts.head().to_html(index=True)))
print(variant_counts.shape)

,barcode,count,library,sample
4618296,CATGAGTGACACAGAG,621,lib1,TiteSeq_01_bin1
4618297,GAACGGGATATCGAAC,540,lib1,TiteSeq_01_bin1
4618298,ATTGAATCACCAAATC,467,lib1,TiteSeq_01_bin1
4618299,AGGCGATTCGTAAATA,390,lib1,TiteSeq_01_bin1
4618300,TGATGAGAGATTCGTG,339,lib1,TiteSeq_01_bin1


(6927444, 4)


In [7]:
codon_variant_table = pd.read_csv(config['codon_variant_table_file'])
display(HTML(codon_variant_table.head().to_html(index=True)))
print(codon_variant_table.shape)

,target,library,barcode,variant_call_support,codon_substitutions,aa_substitutions,n_codon_substitutions,n_aa_substitutions
0,CGG_naive,lib1,AAAAAAAAAACACCGG,6,GGC119GGT TTA200ACT,L200T,2,1
1,CGG_naive,lib1,AAAAAAAAAACATGAG,1,CAG16TGG,Q16W,1,1
2,CGG_naive,lib1,AAAAAAAAAAGCGACG,1,GTG156CAT,V156H,1,1
3,CGG_naive,lib1,AAAAAAAAAAGGAAAG,6,GTG110GGT,V110G,1,1
4,CGG_naive,lib1,AAAAAAAAAATATAGA,1,TAC47CCA,Y47P,1,1


(192429, 8)


Define concentrations and bins

In [8]:
concs = np.array(config['concentrations']['CGG']).astype(float)
concs

array([1.e-06, 1.e-07, 1.e-08, 1.e-09, 1.e-10, 1.e-11, 1.e-12, 1.e-13,
       0.e+00])

Combine tables to make a single barcode level frame

In [9]:
df_barcodes = variant_counts.merge(codon_variant_table, on=("barcode", "library"), how="inner")
display(HTML(df_barcodes.head().to_html(index=True)))
print(df_barcodes.shape)

,barcode,count,library,sample,target,variant_call_support,codon_substitutions,aa_substitutions,n_codon_substitutions,n_aa_substitutions
0,CATGAGTGACACAGAG,621,lib1,TiteSeq_01_bin1,CGG_naive,12,TTT40CTT GCG161AGA,F40L A161R,2,2
1,GAACGGGATATCGAAC,540,lib1,TiteSeq_01_bin1,CGG_naive,50,TAT33TCT,Y33S,1,1
2,ATTGAATCACCAAATC,467,lib1,TiteSeq_01_bin1,CGG_naive,12,TAC218ATG,Y218M,1,1
3,AGGCGATTCGTAAATA,390,lib1,TiteSeq_01_bin1,CGG_naive,19,TGG34AAT,W34N,1,1
4,TGATGAGAGATTCGTG,339,lib1,TiteSeq_01_bin1,CGG_naive,32,CCG9GTT TAC218GTT,P9V Y218V,2,2


(6927444, 10)


In [10]:
# parse the sample name to get the antigen concentration and bin
df_barcodes["antigen_concentration"] = concs[df_barcodes["sample"].str.extract(r"TiteSeq_(\d+)").astype(int) - 1]
df_barcodes["bin"] = df_barcodes["sample"].str[-1].astype(int)

# drop columns that are not needed
df_barcodes.drop(columns=["codon_substitutions", "n_codon_substitutions", "target", "variant_call_support"], inplace=True)
df_barcodes.rename(columns={"aa_substitutions": "variant",
                            "count": "read_count"},
                   inplace=True)
df_barcodes = df_barcodes.loc[:, ["sample", "library", "variant", "n_aa_substitutions", "barcode", "antigen_concentration", "bin", "read_count"]]
df_barcodes.sort_values(by=list(df_barcodes.columns), inplace=True)
df_barcodes.variant = df_barcodes.variant.fillna("WT")

df_barcodes

,sample,library,variant,n_aa_substitutions,barcode,antigen_concentration,bin,read_count
2471,TiteSeq_01_bin1,lib1,A104C,1,AAAAACATCAGTTGGT,0.000001,1,0
2879,TiteSeq_01_bin1,lib1,A104C,1,AAAACACTATCTAGGA,0.000001,1,0
3732,TiteSeq_01_bin1,lib1,A104C,1,AAAATTCAAAATTATC,0.000001,1,0
6540,TiteSeq_01_bin1,lib1,A104C,1,AACAAAAGTGTATGTT,0.000001,1,0
11053,TiteSeq_01_bin1,lib1,A104C,1,AAGTTATGAATACCCT,0.000001,1,0
...,...,...,...,...,...,...,...,...
6927397,TiteSeq_09_bin4,lib2,WT,0,TTTTTCATGTATATGC,0.000000,4,0
6927432,TiteSeq_09_bin4,lib2,WT,0,TTTTTTAAAGTTCATA,0.000000,4,0
6927435,TiteSeq_09_bin4,lib2,WT,0,TTTTTTACCTTTACCT,0.000000,4,0
6927437,TiteSeq_09_bin4,lib2,WT,0,TTTTTTAGAAGCGAAG,0.000000,4,0


In [11]:
barcode_runs.concentration = concs[barcode_runs.concentration.astype(int) - 1]
barcode_runs

sample_type  sort_bin  concentration    date  \
library sample                                                         
lib1    TiteSeq_01_bin1     TiteSeq         1   1.000000e-06  210624   
        TiteSeq_01_bin2     TiteSeq         2   1.000000e-06  210624   
        TiteSeq_01_bin3     TiteSeq         3   1.000000e-06  210624   
        TiteSeq_01_bin4     TiteSeq         4   1.000000e-06  210624   
        TiteSeq_02_bin1     TiteSeq         1   1.000000e-07  210624   
...                             ...       ...            ...     ...   
lib2    TiteSeq_08_bin4     TiteSeq         4   1.000000e-13  210624   
        TiteSeq_09_bin1     TiteSeq         1   0.000000e+00  210624   
        TiteSeq_09_bin2     TiteSeq         2   0.000000e+00  210624   
        TiteSeq_09_bin3     TiteSeq         3   0.000000e+00  210624   
        TiteSeq_09_bin4     TiteSeq         4   0.000000e+00  210624   

                         number_cells  
library sample                         
lib1    TiteSeq_01_bin1          9967  
        TiteSeq_01_bin2        100694  
        TiteSeq_01_bin3        910229  
        TiteSeq_01_bin4       4152079  
        TiteSeq_02_bin1         78772  
...                               ...  
lib2    TiteSeq_08_bin4            23  
        TiteSeq_09_bin1       5016672  
        TiteSeq_09_bin2         86684  
        TiteSeq_09_bin3            50  
        TiteSeq_09_bin4             3  

[72 rows x 5 columns]

Use total cell counts and total read counts in each concentration and bin to estimate the number of cells with each barcode

In [12]:
def normalize_read_count(df):
    library = df.library.iloc[0]
    sample = df["sample"].iloc[0]
    total_reads = df.read_count.sum()
    total_cells = barcode_runs.number_cells[(library, sample)]
    df["estimated_cell_count"] = total_cells * df.read_count / total_reads
    return df

df_barcodes = df_barcodes.groupby(["library", "sample"]).apply(normalize_read_count).reset_index(drop=True)
df_barcodes

,sample,library,variant,n_aa_substitutions,barcode,antigen_concentration,bin,read_count,estimated_cell_count
0,TiteSeq_01_bin1,lib1,A104C,1,AAAAACATCAGTTGGT,0.000001,1,0,0.0
1,TiteSeq_01_bin1,lib1,A104C,1,AAAACACTATCTAGGA,0.000001,1,0,0.0
2,TiteSeq_01_bin1,lib1,A104C,1,AAAATTCAAAATTATC,0.000001,1,0,0.0
3,TiteSeq_01_bin1,lib1,A104C,1,AACAAAAGTGTATGTT,0.000001,1,0,0.0
4,TiteSeq_01_bin1,lib1,A104C,1,AAGTTATGAATACCCT,0.000001,1,0,0.0
...,...,...,...,...,...,...,...,...,...
6927439,TiteSeq_09_bin4,lib2,WT,0,TTTTTCATGTATATGC,0.000000,4,0,0.0
6927440,TiteSeq_09_bin4,lib2,WT,0,TTTTTTAAAGTTCATA,0.000000,4,0,0.0
6927441,TiteSeq_09_bin4,lib2,WT,0,TTTTTTACCTTTACCT,0.000000,4,0,0.0
6927442,TiteSeq_09_bin4,lib2,WT,0,TTTTTTAGAAGCGAAG,0.000000,4,0,0.0


## Barcode aggregation

The current $K_D$ estimation procedure does a separate estimate for each barcode, then computes the median of the estimated $\log K_D$ across barcodes for each variant.
We should instead estimate a single $K_D$ parameter for each variants. We will do this by aggregating read counts from all barcodes for a given variant.

In [13]:
df_variants = df_barcodes.groupby(["library", "variant", "n_aa_substitutions", "antigen_concentration", "bin"]).agg({"read_count": "sum",
                                                                                                                     "estimated_cell_count": "sum",
                                                                                                                     "barcode": "count"}).reset_index()
df_variants.sort_values(by=list(df_variants.columns), inplace=True)
df_variants

,library,variant,n_aa_substitutions,antigen_concentration,bin,read_count,estimated_cell_count,barcode
0,lib1,A104C,1,0.000000e+00,1,2663,1245.976128,25
1,lib1,A104C,1,0.000000e+00,2,25,24.182027,25
2,lib1,A104C,1,0.000000e+00,3,0,0.000000,25
3,lib1,A104C,1,0.000000e+00,4,0,0.000000,25
4,lib1,A104C,1,1.000000e-13,1,578,1488.709081,25
...,...,...,...,...,...,...,...,...
821551,lib2,Y94W R145M,2,1.000000e-07,4,26,15.382275,1
821552,lib2,Y94W R145M,2,1.000000e-06,1,0,0.000000,1
821553,lib2,Y94W R145M,2,1.000000e-06,2,0,0.000000,1
821554,lib2,Y94W R145M,2,1.000000e-06,3,0,0.000000,1


We filter missing concentrations for each barcode in the barcode data, and for each variant in the variant data. Missing means the sum of reads is zero.

In [14]:
def conc_filter_fn(df):
    return all(df.groupby("antigen_concentration").read_count.sum() > 0)

In [15]:
df_barcodes = df_barcodes.groupby(["library", "variant", "n_aa_substitutions", "barcode"]).filter(conc_filter_fn)
df_variants = df_variants.groupby(["library", "variant", "n_aa_substitutions"]).filter(conc_filter_fn)

In [16]:
df_barcodes.to_csv(config['prepped_barcode_counts_file'], index=False)
df_variants.to_csv(config['prepped_variant_counts_file'], index=False)